# House Prices - Regression Models

Xây dựng pipeline hồi quy chuẩn bằng scikit-learn và MLP bằng PyTorch, kèm thực nghiệm feature engineering và chọn đặc trưng.

## 1. Nạp thư viện và đọc dữ liệu

In [14]:
from pathlib import Path
import random
import sys
import numpy as np
import pandas as pd

# Cố định seed để các lần chia dữ liệu và huấn luyện cho kết quả ổn định.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

# Kiểm tra kích thước dữ liệu và tổng số giá trị còn thiếu trong train.
print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Missing values in train:', int(train.isna().sum().sum()))

Train shape: (1460, 81)
Test shape: (1459, 80)
Missing values in train: 7829


## 2. Pipeline tiền xử lý chuẩn

Dùng `ColumnTransformer` để xử lý numeric/categorical. Imputer và encoder chỉ được `fit` trên tập train nhằm tránh data leakage.

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Đọc dữ liệu gốc.
raw_train = pd.read_csv('data/train.csv')
raw_test = pd.read_csv('data/test.csv')
X_raw = raw_train.drop(columns=['SalePrice', 'Id'])
y = raw_train['SalePrice']
X_test_raw = raw_test.drop(columns=['Id'])


def add_engineered_features(data):
    """Tạo các biến tổng hợp có ý nghĩa với giá nhà."""
    features = data.copy()
    features['TotalSF'] = features['TotalBsmtSF'] + features['1stFlrSF'] + features['2ndFlrSF']
    features['TotalBathrooms'] = (
        features['FullBath'] + 0.5 * features['HalfBath']
        + features['BsmtFullBath'] + 0.5 * features['BsmtHalfBath']
    )
    features['Total_sqr_footage'] = (
        features['BsmtFinSF1'] + features['BsmtFinSF2']
        + features['1stFlrSF'] + features['2ndFlrSF']
    )
    features['Total_porch_sf'] = (
        features['OpenPorchSF'] + features['3SsnPorch']
        + features['EnclosedPorch'] + features['ScreenPorch'] + features['WoodDeckSF']
    )
    # Tuổi nhà tại thời điểm bán thường có quan hệ với giá.
    features['HouseAge'] = features['YrSold'] - features['YearBuilt']
    features['RemodAge'] = features['YrSold'] - features['YearRemodAdd']
    features['GarageAge'] = features['YrSold'] - features['GarageYrBlt']
    # Interaction mô tả nhà vừa rộng vừa có chất lượng cao.
    features['OverallQual_GrLivArea'] = features['OverallQual'] * features['GrLivArea']
    features['OverallQual_TotalSF'] = features['OverallQual'] * features['TotalSF']
    return features.replace([np.inf, -np.inf], np.nan)


def make_preprocessor(data):
    """Xử lý missing, categorical và numeric mà không làm leakage."""
    categorical_columns = data.select_dtypes(include=['str']).columns.tolist()
    numeric_columns = data.select_dtypes(exclude=['str']).columns.tolist()
    return ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_columns),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), categorical_columns),
    ])

# Chia validation trước khi fit preprocessing.
X = add_engineered_features(X_raw)
X_test = add_engineered_features(X_test_raw)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)
preprocessor = make_preprocessor(X_train)
X_train_encoded = preprocessor.fit_transform(X_train).astype('float32')
X_valid_encoded = preprocessor.transform(X_valid).astype('float32')
X_test_encoded = preprocessor.transform(X_test).astype('float32')


def regression_report(name, actual, predicted):
    return {
        'model': name,
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
        'R2': r2_score(actual, predicted),
    }

print('Engineered features:', X.shape[1])
print('Encoded features:', X_train_encoded.shape[1])

Engineered features: 88
Encoded features: 294


## 3. Thực nghiệm feature engineering và chọn đặc trưng

So sánh raw features, engineered features và engineered features sau `SelectKBest` trên cùng validation split.

In [16]:
from sklearn.feature_selection import SelectKBest, mutual_info_regression


def evaluate_feature_variant(name, train_features, valid_features, train_target, valid_target):
    """Đánh giá từng nhóm feature bằng cùng một Random Forest và validation split."""
    variant_preprocessor = make_preprocessor(train_features)
    train_encoded = variant_preprocessor.fit_transform(train_features).astype('float32')
    valid_encoded = variant_preprocessor.transform(valid_features).astype('float32')
    model = RandomForestRegressor(
        n_estimators=250,
        max_features=0.8,
        random_state=SEED,
        n_jobs=-1,
    )
    # log1p giảm ảnh hưởng của các căn nhà có giá cực lớn.
    model.fit(train_encoded, np.log1p(train_target))
    predictions = np.expm1(model.predict(valid_encoded))
    report = regression_report(name, valid_target, predictions)
    return report, train_encoded, valid_encoded, variant_preprocessor

# Dùng đúng cùng random_state để raw và engineered có validation rows tương ứng.
X_raw_train, X_raw_valid, y_raw_train, y_raw_valid = train_test_split(
    X_raw, y, test_size=0.2, random_state=SEED
)
raw_report, _, _, _ = evaluate_feature_variant(
    'Raw features', X_raw_train, X_raw_valid, y_raw_train, y_raw_valid
)
engineered_report, _, _, _ = evaluate_feature_variant(
    'Engineered features', X_train, X_valid, y_train, y_valid
)

# Mutual information đo mức liên hệ giữa từng encoded feature và log-price.
# Chỉ giữ 120 feature tốt nhất để giảm nhiễu và giảm số chiều đầu vào.
selector = SelectKBest(mutual_info_regression, k=min(120, X_train_encoded.shape[1]))
selected_train = selector.fit_transform(X_train_encoded, np.log1p(y_train))
selected_valid = selector.transform(X_valid_encoded)
selected_model = RandomForestRegressor(
    n_estimators=250,
    max_features=0.8,
    random_state=SEED,
    n_jobs=-1,
)
selected_model.fit(selected_train, np.log1p(y_train))
selected_predictions = np.expm1(selected_model.predict(selected_valid))
selected_report = regression_report('Engineered + SelectKBest', y_valid, selected_predictions)

feature_experiment_results = pd.DataFrame([
    raw_report,
    engineered_report,
    selected_report,
]).sort_values('RMSE').reset_index(drop=True)
print('Selected features:', int(selector.get_support().sum()))
feature_experiment_results

Selected features: 120


,model,MAE,RMSE,R2
0,Engineered + SelectKBest,16854.584767,28720.047770,0.892463
1,Engineered features,16829.061959,28740.932676,0.892307
2,Raw features,16991.998240,29301.954582,0.888062


## 4. Mô hình chuẩn bằng scikit-learn

Random Forest dùng 250 cây và 120 đặc trưng được chọn, huấn luyện trên `log1p(SalePrice)`.

In [17]:
# Dùng đúng cấu hình tốt nhất đã tìm được ở cell thực nghiệm feature.
sklearn_model = RandomForestRegressor(
    n_estimators=250,
    max_features=0.8,
    random_state=SEED,
    n_jobs=-1,
)
# Huấn luyện trên log-price rồi đổi ngược về đơn vị USD bằng expm1.
sklearn_model.fit(selected_train, np.log1p(y_train))
sklearn_valid_pred = np.expm1(sklearn_model.predict(selected_valid))
results = [regression_report('sklearn RandomForest + SelectKBest', y_valid, sklearn_valid_pred)]
pd.DataFrame(results)

,model,MAE,RMSE,R2
0,sklearn RandomForest + SelectKBest,16854.584767,28720.04777,0.892463


## 5. MLP hồi quy bằng PyTorch

MLP dùng engineered features, có chuẩn hóa, Dropout, AdamW và early stopping.

In [18]:
import copy
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Cố định seed PyTorch để trọng số khởi tạo và kết quả có thể tái lập.
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class HousePriceMLP(nn.Module):
    """MLP hồi quy: nhiều lớp fully-connected cho dữ liệu sau one-hot encoding."""
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 1),
        )

    def forward(self, features):
        return self.network(features).squeeze(1)

# PyTorch cần tensor float32; target cũng học trên log-price giống Random Forest.
X_train_tensor = torch.tensor(X_train_encoded, dtype=torch.float32)
X_valid_tensor = torch.tensor(X_valid_encoded, dtype=torch.float32)
y_train_tensor = torch.tensor(np.log1p(y_train.to_numpy()), dtype=torch.float32)
y_valid_tensor = torch.tensor(np.log1p(y_valid.to_numpy()), dtype=torch.float32)
train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=64,
    shuffle=True,
)

mlp_model = HousePriceMLP(X_train_encoded.shape[1]).to(device)
optimizer = torch.optim.AdamW(mlp_model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_function = nn.MSELoss()
best_state = None
best_valid_loss = float('inf')
stale_epochs = 0
history = {'train_loss': [], 'valid_loss': []}

for epoch in range(300):
    mlp_model.train()
    train_losses = []
    for features, targets in train_loader:
        features, targets = features.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = loss_function(mlp_model(features), targets)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    # Validation chỉ dùng để theo dõi, không cập nhật gradient.
    mlp_model.eval()
    with torch.no_grad():
        valid_loss = loss_function(
            mlp_model(X_valid_tensor.to(device)), y_valid_tensor.to(device)
        ).item()
    history['train_loss'].append(float(np.mean(train_losses)))
    history['valid_loss'].append(valid_loss)

    # Lưu checkpoint tốt nhất và dừng nếu 40 epoch không cải thiện.
    if valid_loss < best_valid_loss - 1e-5:
        best_valid_loss = valid_loss
        best_state = copy.deepcopy(mlp_model.state_dict())
        stale_epochs = 0
    else:
        stale_epochs += 1
        if stale_epochs >= 40:
            break

mlp_model.load_state_dict(best_state)
mlp_model.eval()
with torch.no_grad():
    mlp_valid_log_pred = mlp_model(X_valid_tensor.to(device)).cpu().numpy()
mlp_valid_pred = np.expm1(mlp_valid_log_pred)
results.append(regression_report('PyTorch MLP', y_valid, mlp_valid_pred))
print(f'Device: {device}; epochs: {len(history["train_loss"])}')
pd.DataFrame(results).sort_values('RMSE')

Device: cpu; epochs: 216


,model,MAE,RMSE,R2
0,sklearn RandomForest + SelectKBest,16854.584767,28720.047770,0.892463
1,PyTorch MLP,25425.673828,57075.144292,0.575302


## 6. Huấn luyện toàn bộ dữ liệu và tạo submission

Sau khi đánh giá validation, fit lại preprocessing và hai mô hình trên toàn bộ train để dự đoán test.

In [19]:
X_full_encoded = preprocessor.fit_transform(X).astype('float32')
X_test_encoded = preprocessor.transform(X_test).astype('float32')
full_target = np.log1p(y.to_numpy())

# Chọn lại 120 feature tốt nhất trên toàn bộ train.
full_selector = SelectKBest(mutual_info_regression, k=min(120, X_full_encoded.shape[1]))
X_full_selected = full_selector.fit_transform(X_full_encoded, full_target)
X_test_selected = full_selector.transform(X_test_encoded)

sklearn_model.fit(X_full_selected, full_target)
sklearn_test_pred = np.expm1(sklearn_model.predict(X_test_selected))

full_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_full_encoded, dtype=torch.float32),
        torch.tensor(full_target, dtype=torch.float32),
    ),
    batch_size=64,
    shuffle=True,
)
full_mlp_model = HousePriceMLP(X_full_encoded.shape[1]).to(device)
full_optimizer = torch.optim.AdamW(full_mlp_model.parameters(), lr=1e-3, weight_decay=1e-4)
for _ in range(len(history['train_loss'])):
    full_mlp_model.train()
    for features, targets in full_loader:
        features, targets = features.to(device), targets.to(device)
        full_optimizer.zero_grad()
        loss = loss_function(full_mlp_model(features), targets)
        loss.backward()
        full_optimizer.step()

full_mlp_model.eval()
with torch.no_grad():
    mlp_test_log_pred = full_mlp_model(
        torch.tensor(X_test_encoded, dtype=torch.float32).to(device)
    ).cpu().numpy()
mlp_test_pred = np.expm1(mlp_test_log_pred)

submission_template = pd.read_csv('data/sample_submission.csv')
submission_sklearn = submission_template.copy()
submission_sklearn['SalePrice'] = np.maximum(sklearn_test_pred, 0)
submission_sklearn.to_csv('outputs/sample_sub_sklearn.csv', index=False)

submission_mlp = submission_template.copy()
submission_mlp['SalePrice'] = np.maximum(mlp_test_pred, 0)
submission_mlp.to_csv('outputs/sample_sub_pytorch_mlp.csv', index=False)
print('Đã lưu sample_sub_sklearn.csv và sample_sub_pytorch_mlp.csv')

Đã lưu sample_sub_sklearn.csv và sample_sub_pytorch_mlp.csv
